In [2]:
# Define some variables here
_NYT_CONNECTION_FILE = "/Users/yt/Desktop/nyt-connections-rag/data/ConnectionsFinalDataset.json"
_DIFFICULTY = 3
_GAME_DATE = ""
_NUMBER_OF_CONNECTIONS = 4
#_GITHUB_TOKEN = "GITHUB_TOKEN"
#_URL =https://models.inference.ai.azure.com
_GitHUB_TOKEN = "OPENROUTER_API_KEY"
_URL ="https://openrouter.ai/api/v1"

In [30]:
# Helper functions for wordnet, evaluating connections, and other related tasks

from nltk.corpus import wordnet as wn

def get_word_synsets(words):
    """Return a dict mapping each word to its unique lemma names from WordNet."""
    result = {}
    for word in words:
        synsets = wn.synsets(word.lower().replace(" ", "_"))
        result[word] = list(dict.fromkeys(
            l.name().replace("_", " ") for s in synsets for l in s.lemmas()
        ))
    return result

## Evaluation metrics for predicted groups vs actual answers

def game_completion_rate(all_predicted, all_actual):
    """All 4 groups exactly correct → puzzle is 'completed'."""
    if not all_predicted:
        return 0.0
    completed = 0
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        pred_sets   = [set(p["words"]) for p in predicted]
        if all(ps in actual_sets for ps in pred_sets):
            completed += 1
    return completed / len(all_predicted)


def overall_accuracy(all_predicted, all_actual):
    """Proportion of predicted groups that exactly match a ground-truth group."""
    total, correct = 0, 0
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        for pred in predicted:
            total += 1
            if set(pred["words"]) in actual_sets:
                correct += 1
    return correct / total if total else 0.0


def partial_accuracy_rate(all_predicted, all_actual):
    """
    Proportion of predicted groups whose best overlap with any ground-truth group
    is exactly 3 or exactly 2 words (i.e. near-correct but not exact).
    Returns dict: {3: rate_for_3_overlap, 2: rate_for_2_overlap}
    """
    total = 0
    counts = {2: 0, 3: 0}
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        for pred in predicted:
            pred_set = set(pred["words"])
            best_overlap = max(len(pred_set & a_set) for a_set in actual_sets)
            total += 1
            if best_overlap in counts:
                counts[best_overlap] += 1
    return {k: v / total if total else 0.0 for k, v in counts.items()}



## Evaluation helpers — run once, then call evaluate_run() in the cells below

from IPython.display import display, HTML

def _badge(text, bg, fg="#fff"):
    return (f'<span style="background:{bg};color:{fg};padding:2px 8px;'
            f'border-radius:4px;font-size:0.85em;font-weight:bold">{text}</span>')

def render_puzzle(idx, total, contest, raw_reply, predicted, actual, scores):
    correct_groups = scores["correct_groups"]
    completed      = scores["completed"]
    group_acc      = scores["group_acc"]
    actual_sets    = [set(a["words"]) for a in actual]

    status_badge = (_badge("✓ SOLVED", "#2e7d32") if completed
                    else _badge(f"✗ {correct_groups}/4", "#c62828"))

    html = []
    html.append('<div style="border:1px solid #555;border-radius:8px;padding:16px;'
                'margin:12px 0;font-family:monospace">')

    # Header
    html.append('<div style="display:flex;align-items:center;gap:12px;margin-bottom:12px">')
    html.append(f'<b style="font-size:1.1em">[{idx}/{total}] {contest}</b>')
    html.append(status_badge)
    html.append(f'<span style="color:#888;font-size:0.85em">'
                f'acc {group_acc:.0%} | partial-3 {scores["partial_3"]:.0%} | partial-2 {scores["partial_2"]:.0%}'
                f'</span>')
    html.append('</div>')

    # Raw reply (collapsible)
    escaped = raw_reply.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    html.append('<details style="margin-bottom:12px">')
    html.append('<summary style="cursor:pointer;color:#90caf9">▶ Raw model reply</summary>')
    html.append(f'<pre style="background:#1e1e1e;color:#d4d4d4;padding:10px;border-radius:6px;'
                f'overflow-x:auto;white-space:pre-wrap;font-size:0.85em;margin-top:6px">{escaped}</pre>')
    html.append('</details>')

    # Side-by-side table
    html.append('<table style="width:100%;border-collapse:collapse;font-size:0.9em">')
    html.append('<tr>'
                '<th style="text-align:left;padding:4px 8px;border-bottom:1px solid #444;width:50%;color:#aaa">Predicted</th>'
                '<th style="text-align:left;padding:4px 8px;border-bottom:1px solid #444;width:50%;color:#aaa">Actual</th>'
                '</tr>')

    for row in range(max(len(predicted), len(actual))):
        html.append('<tr style="vertical-align:top">')

        if row < len(predicted):
            pred = predicted[row]
            ps = set(pred["words"])
            is_correct = ps in actual_sets
            bg = "#1b3a1b" if is_correct else "#3a1a1a"
            icon_color = "#66bb6a" if is_correct else "#ef5350"
            icon = "✓" if is_correct else "✗"
            label = pred.get("answerDescription", "?")
            if is_correct:
                words_display = [f'<span style="color:#66bb6a">{w}</span>' for w in pred["words"]]
                note = ""
            else:
                best_a = max(actual, key=lambda a: len(ps & set(a["words"])))
                best_set = set(best_a["words"])
                words_display = [
                    f'<span style="color:{"#66bb6a" if w in best_set else "#ef5350"}">{w}</span>'
                    for w in pred["words"]
                ]
                overlap = len(ps & best_set)
                note = (f'<div style="color:#888;font-size:0.8em;margin-top:2px">'
                        f'best match: <i>{best_a["answerDescription"]}</i> ({overlap}/4)</div>')
            html.append(
                f'<td style="padding:5px 8px;background:{bg};border-radius:4px">'
                f'<span style="color:{icon_color}">{icon}</span> '
                f'<b style="color:#ccc">{label}</b><br>'
                f'{", ".join(words_display)}{note}</td>'
            )
        else:
            html.append('<td></td>')

        if row < len(actual):
            a = actual[row]
            diff_colors = {"Yellow": "#f9a825", "Green": "#2e7d32",
                           "Blue": "#1565c0", "Purple": "#6a1b9a"}
            cat_color = diff_colors.get(a.get("difficulty", ""), "#555")
            html.append(
                f'<td style="padding:5px 8px;background:#1e1e2e;border-radius:4px">'
                f'<b style="color:{cat_color}">{a["answerDescription"]}</b><br>'
                f'<span style="color:#ccc">{", ".join(a["words"])}</span></td>'
            )
        else:
            html.append('<td></td>')

        html.append('</tr>')

    html.append('</table></div>')
    return "".join(html)


def evaluate_run( raw_replies, all_actual):
    """Evaluate a list of raw model replies against ground-truth answers. Returns per_puzzle_scores."""
    display(HTML(
        f'<h2 style="font-family:monospace;border-bottom:2px solid #555;padding-bottom:6px">'
    ))

    predicted_all  = []
    per_puzzle_scores = []

    for i, (raw_reply, actual) in enumerate(zip(raw_replies, all_actual)):
        try:
            s = raw_reply.find('[')
            e = raw_reply.rfind(']') + 1
            predicted = json.loads(raw_reply[s:e])
        except Exception as ex:
            print(f"[{i+1}] Failed to parse JSON: {ex}")
            predicted = []
        predicted_all.append(predicted)

        actual_sets = [set(a["words"]) for a in actual]
        pred_sets   = [set(p["words"]) for p in predicted]

        correct_groups = sum(1 for ps in pred_sets if ps in actual_sets)
        completed      = int(all(ps in actual_sets for ps in pred_sets) and len(pred_sets) == 4)
        partial_3 = sum(
            1 for ps in pred_sets
            if max((len(ps & a) for a in actual_sets), default=0) >= 3
        ) / max(len(pred_sets), 1)
        partial_2 = sum(
            1 for ps in pred_sets
            if max((len(ps & a) for a in actual_sets), default=0) >= 2
        ) / max(len(pred_sets), 1)
        group_acc = correct_groups / max(len(pred_sets), 1)

        scores = dict(correct_groups=correct_groups, completed=completed,
                      group_acc=group_acc, partial_3=partial_3, partial_2=partial_2)
        per_puzzle_scores.append({"contest": pool[i]["contest"], **scores})
        display(HTML(render_puzzle(i + 1, len(pool), pool[i]["contest"],
                                   raw_reply, predicted, actual, scores)))

    # Summary table
    n = len(per_puzzle_scores)
    avg_gcr = sum(s["completed"]  for s in per_puzzle_scores) / n
    avg_acc = sum(s["group_acc"]  for s in per_puzzle_scores) / n
    avg_p3  = sum(s["partial_3"]  for s in per_puzzle_scores) / n
    avg_p2  = sum(s["partial_2"]  for s in per_puzzle_scores) / n

    header = (
        '<tr style="color:#aaa;border-bottom:1px solid #555">'
        '<th style="padding:4px 12px;text-align:left">Contest</th>'
        '<th style="padding:4px 12px">Solved</th>'
        '<th style="padding:4px 12px">Group Acc</th>'
        '<th style="padding:4px 12px">Partial-3</th>'
        '<th style="padding:4px 12px">Partial-2</th></tr>'
    )
    rows = "".join(
        f'<tr><td style="padding:4px 12px">{s["contest"]}</td>'
        f'<td style="padding:4px 12px;text-align:center">{"✓" if s["completed"] else "✗"}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["group_acc"]:.0%}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["partial_3"]:.0%}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["partial_2"]:.0%}</td></tr>'
        for s in per_puzzle_scores
    )
    avg_row = (
        f'<tr style="border-top:2px solid #666;font-weight:bold">'
        f'<td style="padding:6px 12px">Average ({n})</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_gcr:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_acc:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_p3:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_p2:.1%}</td></tr>'
    )
    display(HTML(
        f'<h3 style="font-family:monospace;margin-top:24px">Summary — {n} puzzle{"s" if n!=1 else ""}</h3>'
        f'<table style="border-collapse:collapse;font-family:monospace;font-size:0.9em">'
        f'{header}{rows}{avg_row}</table>'
    ))

    return per_puzzle_scores, dict(gcr=avg_gcr, acc=avg_acc, p3=avg_p3, p2=avg_p2)



In [4]:
import nltk
nltk.download('wordnet')
from nltk.corpus import wordnet as wn


# {
#     "date": "2024/06/03",
#     "contest": "NYT Connections 358 - June 3rd, 2024",
#     "words": [
#         "LASER",
#         "PLUCK",
#         "THREAD",
#         "WAX",
#         "COIL",
#         "SPOOL",
#         "WIND",
#         "WRAP",
#         "HONEYCOMB",
#         "ORGANISM",
#         "SOLAR PANEL",
#         "SPREADSHEET",
#         "BALL",
#         "MOVIE",
#         "SCHOOL",
#         "VITAMIN"
#     ],
#     "answers": [
#         {
#             "answerDescription": "REMOVE, AS BODY HAIR",
#             "words": [
#                 "LASER",
#                 "PLUCK",
#                 "THREAD",
#                 "WAX"
#             ]
#         },
#         {
#             "answerDescription": "TWIST AROUND",
#             "words": [
#                 "COIL",
#                 "SPOOL",
#                 "WIND",
#                 "WRAP"
#             ]
#         },
#         {
#             "answerDescription": "THINGS MADE OF CELLS",
#             "words": [
#                 "HONEYCOMB",
#                 "ORGANISM",
#                 "SOLAR PANEL",
#                 "SPREADSHEET"
#             ]
#         },
#         {
#             "answerDescription": "B-___",
#             "words": [
#                 "BALL",
#                 "MOVIE",
#                 "SCHOOL",
#                 "VITAMIN"
#             ]
#         }
#     ],
#     "difficulty": 3.3
# },

laser = wn.synset('laser.n.01')

[nltk_data] Downloading package wordnet to /Users/yt/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [5]:
# "LASER",
# "PLUCK",
# "THREAD",
# "WAX"

print(wn.synsets('laser')[0].definition())
len(wn.synsets('laser')[0].lemmas())

print(sorted(laser.hypernyms()))

pluck = wn.synset('pluck.v.01')
thread = wn.synset('thread.n.01')
wax = wn.synset('wax.n.01')

lpluck = laser.path_similarity(pluck)
lthread = laser.path_similarity(thread)
lwax = laser.path_similarity(wax)

print("laser-pluck:", lpluck)
print("laser-thread:", lthread)
print("laser-wax:", lwax)

pthread = pluck.path_similarity(thread)
pwax = pluck.path_similarity(wax)

print("pluck-thread:", pthread)
print("pluck-wax:", pwax)

twax = thread.path_similarity(wax)

print("thread-wax:", twax)


an acronym for light amplification by stimulated emission of radiation; an optical device that produces an intense monochromatic beam of coherent light
[Synset('optical_device.n.01')]
laser-pluck: 0.07692307692307693
laser-thread: 0.125
laser-wax: 0.07142857142857142
pluck-thread: 0.08333333333333333
pluck-wax: 0.06666666666666667
thread-wax: 0.07692307692307693


In [19]:
from nltk.corpus import wordnet_ic
nltk.download('wordnet_ic')
brown_ic = wordnet_ic.ic('ic-brown.dat')
semcor_ic = wordnet_ic.ic('ic-semcor.dat')
from nltk.corpus import genesis
nltk.download('genesis')
genesis_ic = wn.ic(genesis, False, 0.0)

# Function for GPT to call WordNet
def analyze_wordnet_relationships(words, analysis_type="similarity"):
    """
    Analyze relationships between words using WordNet.

    Args:
        words: List of words to analyze
        analysis_type: Type of analysis ("similarity", "hypernyms", "definitions")

    Returns:
        Dictionary with analysis results
    """
    results = {}

    if analysis_type == "similarity":
        # Calculate pairwise similarities
        similarities = {}
        for i, word1 in enumerate(words):
            for j, word2 in enumerate(words):
                if i < j:
                    try:
                        syn1 = wn.synsets(word1.lower())[0] if wn.synsets(word1.lower()) else None
                        syn2 = wn.synsets(word2.lower())[0] if wn.synsets(word2.lower()) else None
                        if syn1 and syn2:
                            sim = syn1.path_similarity(syn2)
                            similarities[f"{word1}-{word2}"] = round(sim, 3) if sim else 0
                    except:
                        similarities[f"{word1}-{word2}"] = 0
        results["similarities"] = similarities

    elif analysis_type == "hypernyms":
        # Get hypernyms for each word
        hypernyms = {}
        for word in words:
            try:
                syn = wn.synsets(word.lower())[0] if wn.synsets(word.lower()) else None
                if syn:
                    hypernyms[word] = [h.name().split('.')[0] for h in syn.hypernyms()[:3]]  # Top 3 hypernyms
                else:
                    hypernyms[word] = []
            except:
                hypernyms[word] = []
        results["hypernyms"] = hypernyms

    elif analysis_type == "definitions":
        # Get definitions
        definitions = {}
        for word in words:
            try:
                syn = wn.synsets(word.lower())[0] if wn.synsets(word.lower()) else None
                if syn:
                    definitions[word] = syn.definition()
                else:
                    definitions[word] = "No definition found"
            except:
                definitions[word] = "Error getting definition"
        results["definitions"] = definitions

    return results

# NYT Connections game data
game_data = {
    "words": [
        "LASER", "PLUCK", "THREAD", "WAX",
        "COIL", "SPOOL", "WIND", "WRAP",
        "HONEYCOMB", "ORGANISM", "SOLAR PANEL", "SPREADSHEET",
        "BALL", "MOVIE", "SCHOOL", "VITAMIN"
    ]
}

[nltk_data] Downloading package wordnet_ic to /Users/yt/nltk_data...
[nltk_data]   Package wordnet_ic is already up-to-date!
[nltk_data] Downloading package genesis to /Users/yt/nltk_data...
[nltk_data]   Package genesis is already up-to-date!


In [20]:
import ipywidgets as widgets
from IPython.display import display, Markdown

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url= _URL,
    api_key=os.getenv(_GitHUB_TOKEN)
)



# Function definitions for GPT
tools = [
    {
        "type": "function",
        "function": {
            "name": "analyze_wordnet_relationships",
            "description": "Analyze semantic relationships between words using WordNet",
            "parameters": {
                "type": "object",
                "properties": {
                    "words": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of words to analyze"
                    },
                    "analysis_type": {
                        "type": "string",
                        "enum": ["similarity", "hypernyms", "definitions"],
                        "description": "Type of analysis to perform"
                    }
                },
                "required": ["words", "analysis_type"]
            }
        }
    }
]

def ask_gpt(messages,prompt):
    # Use a fresh local conversation each call, keeping only the system prompt
    local_messages = [messages[0], {"role": "user", "content": prompt}]
    
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=local_messages,
        tools=tools,
        tool_choice="auto"
    )

    if completion.choices and len(completion.choices) > 0:
        message = completion.choices[0].message
        tool_calls = getattr(message, "tool_calls", None)

        if tool_calls:
            # Append the assistant's reply (with tool_calls) to the conversation
            local_messages.append(message)

            for tool_call in tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                if tool_name == "analyze_wordnet_relationships":
                    tool_result = analyze_wordnet_relationships(**tool_args)
                else:
                    tool_result = {"error": f"Unknown tool: {tool_name}"}

                # tool_call_id is required to match this result to the right call
                local_messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result)
                })

            followup = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=local_messages
            )
            if followup.choices:
                return followup.choices[0].message.content

        return getattr(message, "content", None)
    return str(completion)


In [28]:
# Select Words:

import json
import random

with open(_NYT_CONNECTION_FILE) as f:
    dataset = json.load(f)

##Filters
DIFFICULTY            = _DIFFICULTY             # Set to None or "" to skip difficulty filter
GAME_DATE             = _GAME_DATE              # Set to None or "" to skip date filter (format: "YYYY/MM/DD")
NUMBER_OF_CONNECTIONS = _NUMBER_OF_CONNECTIONS  # Number of puzzles to randomly sample; set to None or 0 to use all

pool = dataset
if DIFFICULTY:
    pool = [p for p in pool if p["difficulty"] == DIFFICULTY]
if GAME_DATE:
    pool = [p for p in pool if p["date"] == GAME_DATE]
if NUMBER_OF_CONNECTIONS:
    pool = random.sample(pool, min(NUMBER_OF_CONNECTIONS, len(pool)))

if not pool:
    raise ValueError(f"No puzzles found for DIFFICULTY={DIFFICULTY!r}, GAME_DATE={GAME_DATE!r}")


print(f"Selected {len(pool)} puzzle(s) of difficulty {DIFFICULTY} and date '{GAME_DATE}' for evaluation.")

#Baseline: no hints provied to GPT
raw_replies_without_hints = []
all_actual_without_hints = []
for i, puzzle in enumerate(pool):
    #message building for GPT prompt
    # System prompt for NYT Connections game
    messages = [
            {"role": "system", "content": f"""You are an expert at solving NYT Connections puzzles. 

        The puzzle has 16 words in 4 groups of 4. Each group shares a common theme. Analyze the relationships to find the groups.

        Current puzzle words: {", ".join(puzzle["words"])}

        Think step by step and use your knowledge to find the groups. 
        Return ONLY a JSON array with exactly 4 objects, each with:
        - "answerDescription": a short ALL-CAPS description of the category
        - "words": a list of exactly 4 words from the input

        Example format:
        [
          {{"answerDescription": "CATEGORY NAME", "words": ["WORD1", "WORD2", "WORD3", "WORD4"]}},
          ...
        ]
        """}
        ]  

    # Test GPT with Baseline (no hints)
    print("\nTesting GPT with Baseline (no hints) grouping...")
    group_prompt = (
        "Group the following 16 words into exactly 4 groups of 4 words each, " + ", ".join(puzzle["words"]) + ". "
        + "and return only the groups as lists: " )
    
    raw_reply = ask_gpt(messages, group_prompt)
    raw_replies_without_hints.append(raw_reply)
    all_actual_without_hints.append(puzzle["answers"])
    print(f"  [{i+1}/{len(pool)}] done: {puzzle['contest']}")


#----------------With Wordnet------------------------------
raw_replies_with_hints = []
all_actual_with_hints = []
for i, puzzle in enumerate(pool):
    #message building for GPT prompt
    # System prompt for NYT Connections game
    messages = [
            {"role": "system", "content": f"""You are an expert at solving NYT Connections puzzles. You have access to WordNet to analyze semantic relationships between words.

        Use the analyze_wordnet_relationships function to:
        - Find similarities between words (analysis_type="similarity")
        - Get hypernyms (broader categories) for words (analysis_type="hypernyms") 
        - Get definitions of words (analysis_type="definitions")

        The puzzle has 16 words in 4 groups of 4. Each group shares a common theme. Analyze the relationships to find the groups.

        Current puzzle words: {", ".join(puzzle["words"])}

        Think step by step and use WordNet data to support your reasoning.
        Return ONLY a JSON array with exactly 4 objects, each with:
        - "answerDescription": a short ALL-CAPS description of the category
        - "words": a list of exactly 4 words from the input

        Example format:
        [
          {{"answerDescription": "CATEGORY NAME", "words": ["WORD1", "WORD2", "WORD3", "WORD4"]}},
          ...
        ]
        """}
        ]
    
    
    
    #------------------------------------------
    # Test the WordNet integration
    print("Testing WordNet analysis... for puzzle:", puzzle["contest"])
    result = analyze_wordnet_relationships(puzzle["words"], "similarity")
    print("Similarities:", result["similarities"])
    

    # Test GPT with WordNet
    print("\nTesting GPT with WordNet grouping...")
    group_prompt = (
        "Group the following 16 words into exactly 4 groups of 4 words each, " + ", ".join(puzzle["words"]) + ". "
        + "and return only the groups as lists: " )
    
    raw_reply = ask_gpt(messages, group_prompt)
    raw_replies_with_hints.append(raw_reply)
    all_actual_with_hints.append(puzzle["answers"])
    print(f"  [{i+1}/{len(pool)}] done: {puzzle['contest']}")


Selected 4 puzzle(s) of difficulty 3 and date '' for evaluation.

Testing GPT with Baseline (no hints) grouping...
  [1/4] done: NYT Connections 174 - December 2nd, 2023

Testing GPT with Baseline (no hints) grouping...
  [2/4] done: NYT Connections 206 - January 3rd, 2024

Testing GPT with Baseline (no hints) grouping...
  [3/4] done: NYT Connections 239 - February 5th, 2024

Testing GPT with Baseline (no hints) grouping...
  [4/4] done: NYT Connections 243 - February 9th, 2024
Testing WordNet analysis... for puzzle: NYT Connections 174 - December 2nd, 2023
Similarities: {'BASKET-BIN': 0.333, 'BASKET-CHEST': 0.091, 'BASKET-HAMPER': 0.167, 'BASKET-BAND': 0.083, 'BASKET-CIRCLE': 0.062, 'BASKET-HOOP': 0.125, 'BASKET-RING': 0.071, 'BASKET-CAP': 0.125, 'BASKET-CHECK': 0.059, 'BASKET-CURB': 0.125, 'BASKET-LIMIT': 0.071, 'BASKET-PAPER': 0.091, 'BASKET-PEARL': 0.111, 'BASKET-TRAFFIC': 0.083, 'BIN-CHEST': 0.091, 'BIN-HAMPER': 0.167, 'BIN-BAND': 0.083, 'BIN-CIRCLE': 0.062, 'BIN-HOOP': 0.125, 'B

In [35]:
## Show Results for Baseline (no hints):
scores_no_hints, summary_no_hints = evaluate_run(
    raw_replies_without_hints,
    all_actual_without_hints,
)

Predicted,Actual
"✓ STORAGE CONTAINERSBASKET, BIN, CHEST, HAMPER","CONTAINERSBASKET, BIN, CHEST, HAMPER"
"✓ SHAPESBAND, CIRCLE, HOOP, RING","CIRCULAR SHAPESBAND, CIRCLE, HOOP, RING"
"✓ LIMITSCAP, CHECK, CURB, LIMIT","RESTRICTCAP, CHECK, CURB, LIMIT"
"✓ ASSOCIATED WITH NBANBA, PAPER, PEARL, TRAFFIC","___ JAMNBA, PAPER, PEARL, TRAFFIC"


Predicted,Actual
"✓ FLYINGFLOAT, FLY, GLIDE, SOAR","MOVE THROUGH THE AIRFLOAT, FLY, GLIDE, SOAR"
"✓ MUSICAL TERMSBUG, MIKE, TAP, WIRE","HIDDEN LISTENING DEVICESBUG, MIKE, TAP, WIRE"
"✓ MARKS AND CHECKSCHECK, MARK, TICK, X","SELECT, AS A BOX ON A FORMCHECK, MARK, TICK, X"
"✓ NUMBERS AND TERMS40, COLE, PAIN, TIP","RAPPERS MINUS FIRST LETTER40, COLE, PAIN, TIP"


Predicted,Actual
"✗ FAMILY RELATIONSHIPSBROTHER, NIECE, HEART, SOULbest match: CITY HOMOPHONES (2/4)","""GIVE ME A BREAK!""BROTHER, LORD, PLEASE, SHEESH"
"✗ RELIGIOUS TITLESLORD, BISHOP, CARDINAL, PASTORbest match: ECCLESIASTICAL TITLES (3/4)","ECCLESIASTICAL TITLESBISHOP, CARDINAL, PASTOR, PRIOR"
"✗ NOBILITYPRIOR, QUEEN, MADONNA, PRINCEbest match: ROCK & ROLL HALL OF FAME INDUCTEES (3/4)","ROCK & ROLL HALL OF FAME INDUCTEESHEART, MADONNA, PRINCE, QUEEN"
"✗ SLANG EXPRESSIONSPLEASE, SHEESH, DELI, ROAMbest match: ""GIVE ME A BREAK!"" (2/4)","CITY HOMOPHONESDELI, NIECE, ROAM, SOUL"


Predicted,Actual
"✓ MUSIC CONTROLSREWIND, SHUFFLE, SKIP, STOP","MUSIC PLAYER BUTTONSREWIND, SHUFFLE, SKIP, STOP"
"✓ AUDIO VISUALAIR, RUN, SCREEN, SHOW","BROADCASTAIR, RUN, SCREEN, SHOW"
"✓ BATHROOM ITEMSBATH, CARD, CURTAIN, PICTURE","THINGS YOU CAN DRAWBATH, CARD, CURTAIN, PICTURE"
"✓ TIME MEASURESEON, ETHER, NET, TOW","NUMBER ANAGRAMSEON, ETHER, NET, TOW"


Contest,Solved,Group Acc,Partial-3,Partial-2
"NYT Connections 174 - December 2nd, 2023",✓,100%,100%,100%
"NYT Connections 206 - January 3rd, 2024",✓,100%,100%,100%
"NYT Connections 239 - February 5th, 2024",✗,0%,50%,100%
"NYT Connections 243 - February 9th, 2024",✓,100%,100%,100%
Average (4),75.0%,75.0%,87.5%,100.0%


In [36]:
## Show Results for WordNet:
scores_with_hints, summary_with_hints = evaluate_run(
    raw_replies_with_hints,
    all_actual_with_hints
)

Predicted,Actual
"✓ CONTAINERSBASKET, BIN, CHEST, HAMPER","CONTAINERSBASKET, BIN, CHEST, HAMPER"
"✓ SHAPESBAND, CIRCLE, HOOP, RING","CIRCULAR SHAPESBAND, CIRCLE, HOOP, RING"
"✓ REGULATIONSCAP, CHECK, CURB, LIMIT","RESTRICTCAP, CHECK, CURB, LIMIT"
"✓ CATEGORIESNBA, PAPER, PEARL, TRAFFIC","___ JAMNBA, PAPER, PEARL, TRAFFIC"


Predicted,Actual
"✓ FLIGHT ACTIONSFLOAT, FLY, GLIDE, SOAR","MOVE THROUGH THE AIRFLOAT, FLY, GLIDE, SOAR"
"✓ INSECTS AND EQUIPMENTBUG, MIKE, TAP, WIRE","HIDDEN LISTENING DEVICESBUG, MIKE, TAP, WIRE"
"✓ MARKS AND SYMBOLSCHECK, MARK, TICK, X","SELECT, AS A BOX ON A FORMCHECK, MARK, TICK, X"
"✓ NUMBERS AND TERMS40, COLE, PAIN, TIP","RAPPERS MINUS FIRST LETTER40, COLE, PAIN, TIP"


Predicted,Actual
"✗ FAMILY RELATIONSBROTHER, NIECE, HEART, PLEASEbest match: ""GIVE ME A BREAK!"" (2/4)","""GIVE ME A BREAK!""BROTHER, LORD, PLEASE, SHEESH"
"✓ CLERGY TITLESBISHOP, CARDINAL, PASTOR, PRIOR","ECCLESIASTICAL TITLESBISHOP, CARDINAL, PASTOR, PRIOR"
"✗ ROYAL TITLESLORD, PRINCE, QUEEN, MADONNAbest match: ROCK & ROLL HALL OF FAME INDUCTEES (3/4)","ROCK & ROLL HALL OF FAME INDUCTEESHEART, MADONNA, PRINCE, QUEEN"
"✗ ACTIVITY AND PLACEDELI, ROAM, SHEESH, SOULbest match: CITY HOMOPHONES (3/4)","CITY HOMOPHONESDELI, NIECE, ROAM, SOUL"


Predicted,Actual
"✓ PLAYBACK CONTROLSREWIND, SHUFFLE, SKIP, STOP","MUSIC PLAYER BUTTONSREWIND, SHUFFLE, SKIP, STOP"
"✓ MEDIA ELEMENTSAIR, RUN, SCREEN, SHOW","BROADCASTAIR, RUN, SCREEN, SHOW"
"✓ HOME ELEMENTSBATH, CARD, CURTAIN, PICTURE","THINGS YOU CAN DRAWBATH, CARD, CURTAIN, PICTURE"
"✓ TIME/SPACE TERMSEON, ETHER, NET, TOW","NUMBER ANAGRAMSEON, ETHER, NET, TOW"


Contest,Solved,Group Acc,Partial-3,Partial-2
"NYT Connections 174 - December 2nd, 2023",✓,100%,100%,100%
"NYT Connections 206 - January 3rd, 2024",✓,100%,100%,100%
"NYT Connections 239 - February 5th, 2024",✗,25%,75%,100%
"NYT Connections 243 - February 9th, 2024",✓,100%,100%,100%
Average (4),75.0%,81.2%,93.8%,100.0%


In [37]:
## Comparison: No Hints vs With Hints
from IPython.display import display, HTML

metrics = ["gcr", "acc", "p3", "p2"]
labels  = ["Game Completion Rate", "Group Accuracy", "Partial-3 Rate", "Partial-2 Rate"]

def _delta_cell(a, b):
    diff = b - a
    color = "#66bb6a" if diff > 0 else ("#ef5350" if diff < 0 else "#aaa")
    arrow = "▲" if diff > 0 else ("▼" if diff < 0 else "—")
    return (f'<td style="padding:6px 14px;text-align:center;color:{color};font-weight:bold">'
            f'{arrow} {abs(diff):.1%}</td>')

header = (
    '<tr style="color:#aaa;border-bottom:1px solid #555">'
    '<th style="padding:6px 14px;text-align:left">Metric</th>'
    '<th style="padding:6px 14px">No Hints (A)</th>'
    '<th style="padding:6px 14px">With Hints (B)</th>'
    '<th style="padding:6px 14px">Δ (B − A)</th>'
    '</tr>'
)
rows = "".join(
    f'<tr>'
    f'<td style="padding:6px 14px">{lbl}</td>'
    f'<td style="padding:6px 14px;text-align:center">{summary_no_hints[m]:.1%}</td>'
    f'<td style="padding:6px 14px;text-align:center">{summary_with_hints[m]:.1%}</td>'
    f'{_delta_cell(summary_no_hints[m], summary_with_hints[m])}'
    f'</tr>'
    for m, lbl in zip(metrics, labels)
)

display(HTML(
    '<h2 style="font-family:monospace;border-bottom:2px solid #555;padding-bottom:6px">'
    'Comparison: No Hints vs With WordNet Hints</h2>'
    '<table style="border-collapse:collapse;font-family:monospace;font-size:0.95em">'
    f'{header}{rows}</table>'
    '<p style="color:#888;font-size:0.85em;font-family:monospace">'
    '▲ = With-hints better &nbsp;|&nbsp; ▼ = No-hints better</p>'
))


Metric,No Hints (A),With Hints (B),Δ (B − A)
Game Completion Rate,75.0%,75.0%,— 0.0%
Group Accuracy,75.0%,81.2%,▲ 6.2%
Partial-3 Rate,87.5%,93.8%,▲ 6.2%
Partial-2 Rate,100.0%,100.0%,— 0.0%
